# Day 58: Integrate the AI and Test It Offline

**Today's question:** how do you put a model inside **one function** so the app still works when the model's
reply is wrong, when it refuses, or when there is no key or no network, and how do you test all of that
**without the internet**?

Worked example: the Spending Summary Assistant from Days 56–57. Section 8 shows the same pattern for an ML
model (option A). Before running a cell marked 🤔, predict what it will show.

## 1. The plan for the one function

```
write_summary(totals, ask=ask_llm)
   1. prompt = build_prompt(totals)       only the totals; ask for JSON
   2. reply  = ask(prompt)                real Claude, or the offline mock
   3. result = parse_summary(reply)       json.loads + field checks -> dict, or None
   4. fallback                            refusal or unusable reply -> the Day 57 template
```

The rest of the app only ever sees a clean dict: `{"summary": ..., "tips": [..., ...], "source": ...}`.

## 2. The core from Day 57 (copied so this notebook runs on its own)

In [ ]:
FIXED_COSTS = {"rent", "utilities"}


def get_amount(pair):
    return pair[1]


def top_categories(totals, n=3, skip_fixed=False):
    """The n biggest categories, biggest first."""
    pairs = []
    for category, total in totals.items():
        if skip_fixed and category in FIXED_COSTS:
            continue
        pairs.append((category, total))
    return sorted(pairs, key=get_amount, reverse=True)[:n]


def template_summary(totals):
    """Day 57's stub. From today it is the FALLBACK when the model's reply is unusable."""
    total = sum(totals.values())
    biggest, biggest_amount = top_categories(totals, n=1)[0]
    return {"summary": f"You spent {total:.2f} in total. The biggest category was {biggest} ({biggest_amount:.2f}).",
            "tips": ["Check your spending once a week.", "Set a weekly limit for your biggest category."]}


september = {"food": 660.87, "rent": 650.0, "transport": 187.37, "entertainment": 156.61,
             "shopping": 108.56, "health": 101.59, "utilities": 64.2}
template_summary(september)

## 3. The LLM helper and a mock written for this app

This is the Day 53 `ask_llm` helper. The only app-specific part is `mock_llm`: it reads the numbers **from the
prompt** and answers in the JSON shape we ask for, so the whole pipeline runs offline.

In [ ]:
import json
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()  # read .env now, so LLM_MODEL can come from the .env file too
MODEL = os.environ.get("LLM_MODEL", "claude-opus-5-5")
REFUSAL_MESSAGE = "[The model declined this request.]"


def mock_llm(prompt):
    """Offline stand-in: summarise the '- category: amount' lines of the prompt as JSON."""
    flexible = []
    total = "?"
    for line in prompt.splitlines():
        if line.startswith("- ") and ": " in line and "(fixed)" not in line:
            category, amount = line[2:].split(": ")
            flexible.append((category, amount))
        elif line.startswith("Total: "):
            total = line[len("Total: "):]
    if not flexible:
        return json.dumps({"summary": f"You spent {total} in total.",
                           "tips": ["Write down every expense for a week.", "Set a simple weekly limit."]})
    top, top_amount = flexible[0]  # the prompt lists categories biggest first
    second = top
    if len(flexible) > 1:
        second = flexible[1][0]
    return json.dumps({
        "summary": f"You spent {total} in total. Your biggest flexible category was {top} at {top_amount}.",
        "tips": [f"Set a weekly limit for {top} and check it every Sunday.",
                 f"Look at {second} next: small savings there add up over a month."],
    })

In [ ]:
def ask_llm(prompt, system="You are a helpful assistant."):
    """Send one prompt to Claude and return its text reply (mock reply when offline)."""
    load_dotenv()  # reads ANTHROPIC_API_KEY from a .env file if present
    if os.environ.get("COURSE_OFFLINE") == "1" or not os.environ.get("ANTHROPIC_API_KEY"):
        return mock_llm(prompt)
    client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY automatically
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=16000,
        system=system,
        messages=[{"role": "user", "content": prompt}],
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",  # if this model declines, the API retries on a suitable fallback model
    )
    if response.stop_reason == "refusal":
        return REFUSAL_MESSAGE
    # The reply is a list of blocks (thinking blocks may come first), so keep only the text blocks.
    return "".join(block.text for block in response.content if block.type == "text")


if os.environ.get("COURSE_OFFLINE") == "1" or not os.environ.get("ANTHROPIC_API_KEY"):
    print("Mode: offline mock")
else:
    print(f"Mode: online ({MODEL})")

## 4. Building the prompt

The prompt starts with the role and its limits (general tips, no financial-product advice). Only category
totals go in: no dates, no descriptions (Day 56's privacy decision). Fixed costs are marked so the tips can
ignore them. We ask for JSON and forbid invented numbers.

In [ ]:
def build_prompt(totals):
    """Prompt with ONLY the category totals, biggest first."""
    lines = []
    for category, amount in top_categories(totals, n=len(totals)):
        note = ""
        if category in FIXED_COSTS:
            note = " (fixed)"
        lines.append(f"- {category}: {amount:.2f}{note}")
    return (
        "You are a friendly budgeting helper for students. Give simple, general budgeting tips,\n"
        "never investment or financial-product advice.\n\n"
        "Here are a student's spending totals for one month, by category, biggest first:\n"
        + "\n".join(lines)
        + f"\nTotal: {sum(totals.values()):.2f}\n\n"
        "Write a summary of at most two sentences and exactly two practical tips.\n"
        "Tips must be about categories that are NOT marked (fixed).\n"
        "Only use numbers that appear above. Do not invent amounts or percentages.\n"
        'Reply with JSON only, in this shape: {"summary": "...", "tips": ["...", "..."]}'
    )

🤔 **Predict:** where in the prompt will `rent` appear, and what is written after its amount?

In [ ]:
print(build_prompt(september))

In [ ]:
reply = ask_llm(build_prompt(september))
print(reply)

(Offline this reply comes from `mock_llm`. With a key, the same cell calls Claude and the wording will differ.)

## 5. Never trust the reply: parse and check it

`json.loads` only proves the text is JSON. We also check the **fields**: a dict, a non-empty `summary`, a list of
exactly two non-empty `tips`. Anything else → `None`, and the caller decides the fallback.

In [ ]:
def parse_summary(reply):
    """Return {"summary": text, "tips": [text, text]}, or None if the reply is not usable."""
    text = reply.strip()
    if text.startswith("```"):  # some models wrap JSON in a ```json ... ``` code fence
        text = text.strip("`")
        if text.startswith("json"):
            text = text[4:]
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return None
    if not isinstance(data, dict):
        return None
    summary = data.get("summary")
    tips = data.get("tips")
    if not isinstance(summary, str) or summary.strip() == "":
        return None
    if not isinstance(tips, list) or len(tips) != 2:
        return None
    for tip in tips:
        if not isinstance(tip, str) or tip.strip() == "":
            return None
    return {"summary": summary.strip(), "tips": [tip.strip() for tip in tips]}

These replies are **hand-written examples** of what models sometimes send back (not real model output).
🤔 **Predict** which ones survive `parse_summary`:

In [ ]:
example_replies = [
    '{"summary": "You spent 1929.20.", "tips": ["Cook more.", "Walk more."]}',
    'Sure! Here is your summary: {"summary": "You spent 1929.20.", "tips": ["Cook more.", "Walk more."]}',
    '```json\n{"summary": "You spent 1929.20.", "tips": ["Cook more.", "Walk more."]}\n```',
    '{"summary": "You spent 1929.20.", "tips": ["Cook more."]}',
    '{"summary": "", "tips": ["Cook more.", "Walk more."]}',
    '["You spent 1929.20.", "Cook more."]',
]
for example in example_replies:
    print(parse_summary(example) is not None, "|", example[:60].replace("\n", " "))

## 6. The one function, with fallbacks

`ask` is a **parameter** whose default is `ask_llm`. A function is a value, so tests can pass a different one
(like `key=` in `sorted`, Day 21). API errors fall back to the mock; refusals and unusable replies fall back to
the template.

In [ ]:
def call_model(prompt, ask):
    """Ask the model. If the API fails (key, limits, network), let the offline mock answer instead."""
    try:
        return ask(prompt)
    except anthropic.AuthenticationError:
        print("(The API key was rejected, so the offline mock answered. Check your .env file.)")
    except anthropic.RateLimitError:
        print("(Too many requests right now, so the offline mock answered.)")
    except anthropic.APIConnectionError:
        print("(No connection to the API, so the offline mock answered.)")
    except anthropic.APIStatusError as error:
        print(f"(The API returned error {error.status_code}, so the offline mock answered.)")
    return mock_llm(prompt)


def write_summary(totals, ask=ask_llm):
    """The ONE AI function. Always returns {"summary", "tips", "source"} and never crashes."""
    reply = call_model(build_prompt(totals), ask)
    if reply == REFUSAL_MESSAGE:
        result = template_summary(totals)
        result["source"] = "template (the model declined)"
        return result
    result = parse_summary(reply)
    if result is None:
        result = template_summary(totals)
        result["source"] = "template (the reply was not valid JSON)"
        return result
    result["source"] = "ai"
    return result


write_summary(september)

Now **fake models**: tiny functions with the same shape as `ask_llm` that return one fixed reply each.
They let us test every branch without a network, a key or any cost.

In [ ]:
def fake_chatty(prompt):
    return "Sure! You spent a lot on food this month."


def fake_refusal(prompt):
    return REFUSAL_MESSAGE


def fake_good(prompt):
    return '{"summary": "Food was your biggest cost.", "tips": ["Cook at home.", "Share rides."]}'

🤔 **Predict** the `source` of each result before running:

In [ ]:
for fake in [fake_good, fake_chatty, fake_refusal]:
    result = write_summary(september, ask=fake)
    print(f"{fake.__name__:<13} -> {result['source']}")

(`fake.__name__` is just the function's name as text, handy for printing.)

**Missing key → mock** is already handled inside `ask_llm`. API errors (wrong key, rate limit, no network) are
caught in `call_model`. We cannot easily create a network error in a notebook, so we test that path once by hand
on Day 59 (Wi-Fi off, real key in `.env`).

## 7. Tests with fake models

In a test file the same idea becomes plain pytest functions. Here we call them by hand to see them pass:

In [ ]:
def test_good_reply_is_used():
    assert write_summary(september, ask=fake_good)["source"] == "ai"


def test_chatty_reply_falls_back_to_template():
    result = write_summary(september, ask=fake_chatty)
    assert result["source"] == "template (the reply was not valid JSON)"
    assert len(result["tips"]) == 2


def test_refusal_falls_back_to_template():
    assert write_summary(september, ask=fake_refusal)["source"] == "template (the model declined)"


test_good_reply_is_used()
test_chatty_reply_falls_back_to_template()
test_refusal_falls_back_to_template()
print("3 tests passed")

The files version is in `example_app/` (Day 57's app + `llm.py`, the new `summary.py` and `test_summary.py`):

```
cd weeks/week-12/day-58-capstone-ai-integration/example_app
uv run pytest -q          # 15 passed, no network needed
uv run python main.py     # AI mode: offline mock (or online with a key in .env)
```

`test_summary.py` also sets `os.environ["COURSE_OFFLINE"] = "1"` at the top, so even a test that uses the
default `ask_llm` can never call the paid API by accident.

## 8. Option A: a model trained at start-up, behind one function

Same pattern without an LLM: train once when the app starts, then the app only calls `predict_price`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

FEATURES = ["area_sqm", "bedrooms", "distance_km"]


def train_price_model(path=Path("../../../datasets/apartments.csv")):
    """Train at start-up. Returns (model, test MAE in thousands)."""
    data = pd.read_csv(path)
    X_train, X_test, y_train, y_test = train_test_split(
        data[FEATURES], data["price_k"], test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model, mean_absolute_error(y_test, model.predict(X_test))


price_model, mae = train_price_model()
print(f"Trained. Typical error on unseen apartments: {mae:.1f}k")

In [ ]:
def predict_price(model, area_sqm, bedrooms, distance_km):
    """The ONE AI function for option A: check the inputs, predict, return a dict."""
    if area_sqm < 15 or area_sqm > 250:
        raise ValueError("area_sqm must be between 15 and 250")
    if bedrooms < 1 or bedrooms > 6 or distance_km < 0 or distance_km > 50:
        raise ValueError("bedrooms must be 1-6 and distance_km 0-50")
    row = pd.DataFrame([[area_sqm, bedrooms, distance_km]], columns=FEATURES)
    return {"price_k": round(float(model.predict(row)[0]), 1)}


print(predict_price(price_model, 70, 2, 5.0))

🤔 **Predict:** if the area goes from 70 to 100 m² (everything else equal), should the price go up or down?
A test can check that **direction** even though it cannot know the exact price:

In [ ]:
small = predict_price(price_model, 70, 2, 5.0)["price_k"]
large = predict_price(price_model, 100, 2, 5.0)["price_k"]
assert large > small
print("bigger apartment -> higher price:", small, "->", large)

For ML capstones, good offline tests are: the data loads with the right columns, the model beats the baseline,
bad inputs raise `ValueError`, and predictions move in a sensible direction.

## 9. A small evaluation: does it meet our success criteria?

Day 56's criterion: *summary ≤ 40 words and names the biggest flexible category, for at least 4 of 5 cases.*
A loop over test cases measures it. Offline this measures the pipeline with the mock; with a key, the **same
loop** measures the real model (5 API calls).

In [ ]:
test_cases = [
    september,
    {"food": 728.59, "rent": 650.0, "transport": 210.01, "shopping": 201.61, "entertainment": 190.12},
    {"rent": 650.0, "food": 359.49, "transport": 235.03, "shopping": 229.92, "health": 214.11},
    {"food": 20.0, "rent": 650.0, "transport": 3.0},
    {"rent": 650.0, "utilities": 81.48, "entertainment": 12.0},
]

passed = 0
for totals in test_cases:
    result = write_summary(totals)
    top_flexible = top_categories(totals, n=1, skip_fixed=True)[0][0]
    short = len(result["summary"].split()) <= 40
    names_top = top_flexible in result["summary"].lower()
    ok = result["source"] == "ai" and short and names_top
    if ok:
        passed += 1
    print(f"{top_flexible:<14} source={result['source']:<3} short={short} names_top={names_top}")
print(f"\nPassed {passed}/{len(test_cases)}")

🤔 **Think:** why does 5/5 with the mock **not** prove the app is good? (We wrote the mock ourselves, so it
always behaves. Offline, the loop proves the *pipeline* works: prompt → parse → checks. The model's quality is
only measured by running the same loop online.)

Write the result in your README as *"5/5 with the mock; N/5 with <model name> on <date>"*, using the numbers
you actually measured. An honest number, with its date and model, is better than "it works well".

## 10. Common mistakes (these cells fail on purpose)

**Mistake 1:** trusting the reply and calling `json.loads` directly.

In [ ]:
data = json.loads(fake_chatty("any prompt"))

**Mistake 2:** *calling* the fake instead of *passing* it. `ask=fake_good()` runs it immediately (without a
prompt), instead of handing over the function.

In [ ]:
write_summary(september, ask=fake_good())

| Mistake | Symptom | Fix |
|---------|---------|-----|
| `json.loads(reply)` without `try` | `JSONDecodeError` crashes the app | parse in a function that returns `None` |
| `ask=fake()` instead of `ask=fake` | `TypeError: ... missing 1 required positional argument` | pass the function, do not call it |
| `response.content[0].text` | crash or empty text: the first block can be a thinking block | join the `text` blocks (as in `ask_llm`) |
| tests use the real API | slow, costs money, fail without network | fake models + `COURSE_OFFLINE=1` in the test file |
| no fallback | one bad reply ruins the demo | refusal / bad JSON → template; API error → mock |
| fake without `return` | `AttributeError: 'NoneType' object has no attribute 'strip'` | a fake returns a string, like a model |

## Recap

1. The AI lives in **one function**: prompt → ask → parse/check → fallback. The app only sees clean dicts.
2. `ask=ask_llm` as a default parameter lets tests pass **fake models**: test every branch offline.
3. Never trust a reply: `json.loads` in `try`, then check every field; `None` means "use the fallback".
4. Option A: train once at start-up; test loading, baseline, bad inputs and the direction of predictions.
5. Measure your success criteria with a small evaluation loop and report the honest number.

**Next time:** reliability and responsible AI: failure cases, hallucination and privacy checks, README and demo script.